In [1]:
!kaggle datasets download -d thoughtvector/customer-support-on-twitter

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:01<00:00, 92.3MB/s]



In [2]:
import zipfile

with zipfile.ZipFile('/content/customer-support-on-twitter.zip', 'r') as zip_ref:
  zip_ref.extractall('/content/')

print("File Unzipped!")

File Unzipped!


In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/twcs/twcs.csv')

# Check the first few rows to understand the data
print(df.head())

   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3                      5.0  
4

In [4]:
# Extract tweets from the 'text' column or any other relevant column
tweets = df['text'].dropna().tolist() # This assumes the column with tweets is named 'text'

In [5]:
# Convert the list of tweets to a DataFrame
df_tweets = pd.DataFrame(tweets, columns=['text'])

# Save the DataFrame to a CSV file
df_tweets.to_csv('tweets.csv', index=False, encoding='utf-8')

In [6]:
# Checking the length of df
formatted_length = "{:,}".format(len(df_tweets))
print(formatted_length)

2,811,774


In [7]:
for tweet in tweets[:10]:  # This will display the first 5 tweets
    print(tweet)

@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.
@sprintcare and how do you propose we do that
@sprintcare I have sent several private messages and no one is responding as usual
@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
@sprintcare I did.
@115712 Can you please send us a private message, so that I can gain further details about your account?
@sprintcare is the worst customer service
@115713 This is saddening to hear. Please shoot us a DM, so that we can look into this for you. -KC
@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯
@115713 We understand your concerns and we'd like for you to please send us a Direct Message, so that we can further assist you. -AA


In [8]:
import re

def filter_tweet(tweet):
  # Keep only characters a to z, spaces, and apostrophes, then convert to lowercase
  return  re.sub(r'[^a-z\s\']', '', tweet.lower())

filtered_tweets = [filter_tweet(tweet) for tweet in tweets]

In [9]:
f = 30
filtered_tweets = [tweet for tweet in filtered_tweets if len(tweet.split()) > f] # Only keep tweets with more then f words

In [10]:
for filtered_tweet in filtered_tweets[:10]:  # This will display the first 5 tweets
    print(filtered_tweet)

marksandspencer i check with the gov office and legal they stated you are not right but its funny how the other stores dont but you do no wonder lidl and the rest are beating you
marksandspencer ou must charge at least p a bag including vat for carrier bags that are all of the following

unused  its new and hasnt already been used for sold goods to be taken away or delivered
plastic and  microns thick or less
it has handles an opening and isnt sealed
marksandspencer arent require charge  a bag
paper bags
shops in airports or on board trains aeroplanes or ships
bags which only contain certain items such as unwrapped food raw meat and fish where there is a food safety risk prescription medicines uncovered blades seeds bulbs amp s
 hi you can change your microsoft account email through the steps here httpstcodkehohboyy  if the email your son wants to change to is already associated with a microsoft account you'll need to follow those steps to switch the email address on that account too z

In [11]:
# Checking the length of dataset
formatted_length = "{:,}".format(len(filtered_tweets))
print(formatted_length)

228,637


In [12]:
import os

if not os.path.exists('/content/model/dataset'):
  os.makedirs('/content/model/dataset')

In [13]:
import csv

# Save to CSV
with open('/content/model/dataset/processed_tweets.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    for tweet in filtered_tweets:
        writer.writerow([tweet])

In [14]:
import csv

# Read from CSV
with open('/content/model/dataset/processed_tweets.csv', 'r') as file:
    reader = csv.reader(file)

    # Use islice from itertools to only get the first 5 lines
    from itertools import islice
    for row in islice(reader, 5):
        print(row[0])

marksandspencer i check with the gov office and legal they stated you are not right but its funny how the other stores dont but you do no wonder lidl and the rest are beating you
marksandspencer ou must charge at least p a bag including vat for carrier bags that are all of the following

unused  its new and hasnt already been used for sold goods to be taken away or delivered
plastic and  microns thick or less
it has handles an opening and isnt sealed
marksandspencer arent require charge  a bag
paper bags
shops in airports or on board trains aeroplanes or ships
bags which only contain certain items such as unwrapped food raw meat and fish where there is a food safety risk prescription medicines uncovered blades seeds bulbs amp s
 hi you can change your microsoft account email through the steps here httpstcodkehohboyy  if the email your son wants to change to is already associated with a microsoft account you'll need to follow those steps to switch the email address on that account too z

In [15]:
#@title Checking that PyTorch Sees CUDA
import torch
torch. cuda.is_available()

False

In [16]:
from transformers import RobertaConfig, RobertaForCausalLM

config = RobertaConfig(
    vocab_size=52000,
    max_position_embeddings=514,
    num_attention_heads=12,
    num_hidden_layers=6,
    type_vocab_size=1,
    is_decoder=True
)

In [17]:
# Create the RobertaForCausalLM model with the specified config
model = RobertaForCausalLM(config=config)
print(model)

RobertaForCausalLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [18]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
print("Special tokens:", tokenizer.special_tokens_map)

Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [20]:
print(model.num_parameters())

83504416


In [21]:
LP=list(model.parameters())
lp=len(LP)
print(lp)
for p in range(0,lp):
  print(LP[p])

106
Parameter containing:
tensor([[-0.0022,  0.0192,  0.0440,  ..., -0.0550,  0.0163, -0.0022],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0182, -0.0123, -0.0440,  ..., -0.0071, -0.0037, -0.0056],
        ...,
        [ 0.0001,  0.0041,  0.0342,  ..., -0.0212,  0.0132, -0.0080],
        [-0.0066,  0.0189,  0.0247,  ..., -0.0050, -0.0371,  0.0308],
        [ 0.0105, -0.0057, -0.0096,  ..., -0.0076, -0.0201,  0.0072]],
       requires_grad=True)
Parameter containing:
tensor([[ 3.1671e-02,  1.2678e-03, -2.2652e-03,  2.2183e-02,  2.0165e-02,
         -2.1251e-03, -1.1217e-02,  1.7513e-02,  1.0890e-02, -4.0072e-02,
         -1.4575e-02, -1.0798e-02, -2.7592e-02, -2.3096e-02, -1.9390e-02,
         -4.4703e-03, -1.6446e-03,  1.4728e-02, -1.0292e-02,  1.3737e-03,
         -3.8374e-03, -5.4212e-03, -8.9874e-03,  8.6270e-03,  1.5623e-02,
         -7.6036e-03,  7.2121e-04,  7.9096e-03, -2.0672e-02, -4.3538e-02,
         -1.5552e-02,  6.4338e-03,  2.4581e-02,

In [22]:
#Shape of each tensor in the model
LP = list(model.parameters())
for i, tensor in enumerate(LP):
    print(f"Shape of tensor {i}: {tensor.shape}")

Shape of tensor 0: torch.Size([52000, 768])
Shape of tensor 1: torch.Size([1, 768])
Shape of tensor 2: torch.Size([768])
Shape of tensor 3: torch.Size([768])
Shape of tensor 4: torch.Size([514, 768])
Shape of tensor 5: torch.Size([768, 768])
Shape of tensor 6: torch.Size([768])
Shape of tensor 7: torch.Size([768, 768])
Shape of tensor 8: torch.Size([768])
Shape of tensor 9: torch.Size([768, 768])
Shape of tensor 10: torch.Size([768])
Shape of tensor 11: torch.Size([768, 768])
Shape of tensor 12: torch.Size([768])
Shape of tensor 13: torch.Size([768])
Shape of tensor 14: torch.Size([768])
Shape of tensor 15: torch.Size([3072, 768])
Shape of tensor 16: torch.Size([3072])
Shape of tensor 17: torch.Size([768, 3072])
Shape of tensor 18: torch.Size([768])
Shape of tensor 19: torch.Size([768])
Shape of tensor 20: torch.Size([768])
Shape of tensor 21: torch.Size([768, 768])
Shape of tensor 22: torch.Size([768])
Shape of tensor 23: torch.Size([768, 768])
Shape of tensor 24: torch.Size([768])
Sh

In [23]:
#counting the parameters
np=0
for p in range(0,lp):#number of tensors
  PL2=True
  try:
    L2=len(LP[p][0]) #check if 2D
  except:
    L2=1             #not 2D but 1D
    PL2=False
  L1=len(LP[p])
  L3=L1*L2
  np+=L3             # number of parameters per tensor
  if PL2==True:
    print(p,L1,L2,L3)  # displaying the sizes of the parameters
  if PL2==False:
    print(p,L1,L3)  # displaying the sizes of the parameters

print(np)              # total number of parameters

0 52000 768 39936000
1 1 768 768
2 768 768
3 768 768
4 514 768 394752
5 768 768 589824
6 768 768
7 768 768 589824
8 768 768
9 768 768 589824
10 768 768
11 768 768 589824
12 768 768
13 768 768
14 768 768
15 3072 768 2359296
16 3072 3072
17 768 3072 2359296
18 768 768
19 768 768
20 768 768
21 768 768 589824
22 768 768
23 768 768 589824
24 768 768
25 768 768 589824
26 768 768
27 768 768 589824
28 768 768
29 768 768
30 768 768
31 3072 768 2359296
32 3072 3072
33 768 3072 2359296
34 768 768
35 768 768
36 768 768
37 768 768 589824
38 768 768
39 768 768 589824
40 768 768
41 768 768 589824
42 768 768
43 768 768 589824
44 768 768
45 768 768
46 768 768
47 3072 768 2359296
48 3072 3072
49 768 3072 2359296
50 768 768
51 768 768
52 768 768
53 768 768 589824
54 768 768
55 768 768 589824
56 768 768
57 768 768 589824
58 768 768
59 768 768 589824
60 768 768
61 768 768
62 768 768
63 3072 768 2359296
64 3072 3072
65 768 3072 2359296
66 768 768
67 768 768
68 768 768
69 768 768 589824
70 768 768
71 768 768

In [24]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset('csv', data_files='/content/model/dataset/processed_tweets.csv', column_names=["text"])

Generating train split: 0 examples [00:00, ? examples/s]

In [25]:
# split datasets into train and eval
from datasets import DatasetDict

dataset = dataset['train'].train_test_split(test_size=0.1)
dataset = DatasetDict(dataset)

In [26]:
def tokenize_function(examples):
  return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/205773 [00:00<?, ? examples/s]

Map:   0%|          | 0/22864 [00:00<?, ? examples/s]

In [27]:
# Define the data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # For causal (autoregressive) language modeling
)

In [28]:
# to display the time every x steps suring training
from transformers import Trainer
from datetime import datetime
from typing import Dict, Any

class CustomTrainer(Trainer):
    # Add *args or start_time=None to handle the extra positional argument
    def log(self, logs: Dict[str, Any], *args, **kwargs) -> None:
        # Pass everything up to the parent class
        super().log(logs, *args, **kwargs)

        # In modern Transformers, the step isn't always inside 'logs'
        # but is available at self.state.global_step
        step = self.state.global_step

        # Ensure eval_steps is defined to avoid division by zero
        eval_steps = self.args.eval_steps if self.args.eval_steps else 500

        if step > 0 and step % eval_steps == 0:
            print(f"Current time at step {step}: {datetime.now()}")

In [38]:
import logging
from transformers import Trainer, TrainingArguments

# Set up Python logging
logging.basicConfig(loevel=logging.INFO)

training_args = TrainingArguments(
    output_dir="/content/model/model/",
    num_train_epochs=2,                  # can be increased to increase accuracy if productive
    per_device_train_batch_size=128,     # batch size per device
    per_device_eval_batch_size=128,      # batch size for evaluation
    warmup_steps=500,                    # number of warmup steps for learning rate scheduler
    weight_decay=0.01,                   # strength of weight decay
    save_steps=10_000,                   # save a checkpoint every save_steps=10000
    save_total_limit=2,                  # the maximum number of checkpoint model files to keep
    logging_dir='/content/model/logs/',  # directory for storing logs
    logging_steps=100,                   # Log every 100 steps
    logging_first_step=True,             # Log the first step
    eval_steps=500,                      # Evaluate every 500 steps
    # fp16=True
    optim="adamw_torch",
    bf16=True,
    fp16=False,
    dataloader_drop_last=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [39]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"]
)

In [40]:
%%time
trainer.train()

Step,Training Loss
1,5.492180
100,5.432670
200,5.390082
300,5.269358
400,5.182075
500,5.070588
600,4.979260
700,4.897324
800,4.835345
900,4.762324


Current time at step 500: 2026-05-08 03:14:29.242969
Current time at step 1000: 2026-05-08 03:15:40.618923
Current time at step 1500: 2026-05-08 03:16:51.784403
Current time at step 2000: 2026-05-08 03:18:03.307390
Current time at step 2500: 2026-05-08 03:19:14.944995
Current time at step 3000: 2026-05-08 03:20:26.611292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CPU times: user 6min 16s, sys: 5.21 s, total: 6min 22s
Wall time: 7min 46s


TrainOutput(global_step=3214, training_loss=4.597746509013149, metrics={'train_runtime': 466.3345, 'train_samples_per_second': 882.182, 'train_steps_per_second': 6.892, 'total_flos': 1.3640435735986176e+16, 'train_loss': 4.597746509013149, 'epoch': 2.0})

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
import torch_xla.core.xla_model as xm

# 1. Get the correct device (TPU)
device = xm.xla_device()
model.to(device)
model.eval() # Set to evaluation mode

# Define the function to generate response
def generate_response(prompt):
    # Move inputs to the TPU device
    inputs = tokenizer(prompt, return_tensors="pt", max_length=50, truncation=True).to(device)

    # RoBERTa needs the pad_token_id explicitly set for generation
    # We also add do_sample=True since you are using temperature
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=200,
            temperature=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text

# --- Widget Section remains mostly the same ---
text_input = widgets.Textarea(
    description='Prompt:',
    placeholder='What is the categorical imperative?',
    layout={'width': '100%'}
)

button = widgets.Button(
    description='Generate',
    button_style='success'
)

output_text = widgets.Output(layout={'border': '1px solid gray', 'padding': '10px', 'margin': '10px 0'})

# Define button click event handler
def on_button_clicked(b):
    with output_text:
        clear_output()
        response = generate_response(text_input.value)
        print(response)

button.on_click(on_button_clicked)

# Display widgets
display(text_input, button, output_text)